## Segunda parte da aula

In [1]:
import os
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

2025-01-14 17:42:02.107247: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-14 17:42:02.282729: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1736890922.365451   19649 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1736890922.393229   19649 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-14 17:42:02.562938: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
path= '/home/turma02/lgacademy/potatochips/potato_chips/'

In [3]:
num_classes = 2

In [4]:
modelo_base = ResNet50(weights='imagenet', include_top = False, input_shape = (224, 224, 3))

I0000 00:00:1736890926.309189   19649 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1207 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [5]:
modelo_novo = modelo_base.output
modelo_novo = GlobalAveragePooling2D()(modelo_novo)
modelo_novo = Dense(1024, activation = 'relu')(modelo_novo)
predicoes = Dense(num_classes, activation = 'softmax')(modelo_novo)

In [6]:
modelo = Model(inputs = modelo_base.input, outputs = predicoes)

In [7]:
for layer in modelo_base.layers:
    layer.trainable = False

In [8]:
modelo.compile(optimizer=Adam(learning_rate=0.001),
               loss = 'categorical_crossentropy',
               metrics=['accuracy'])

In [9]:
train_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    rotation_range = 40,
    width_shift_range = 0.2,
    height_shift_range = 0.2,
    shear_range = 0.2,
    zoom_range = 0.2,
    horizontal_flip = True,
    fill_mode = 'nearest'
)

In [10]:
validation_datagen = ImageDataGenerator(preprocessing_function = preprocess_input)

In [11]:
train_generator = train_datagen.flow_from_directory(
    os.path.join(path, 'Train'),
    target_size = (224, 224),
    batch_size = 10,
    class_mode = 'categorical'
)

Found 769 images belonging to 2 classes.


In [12]:
validation_generator = validation_datagen.flow_from_directory(
    os.path.join(path, 'Test'),
    target_size = (224, 224),
    batch_size = 10,
    class_mode = 'categorical'
)

Found 176 images belonging to 2 classes.


In [13]:
modelo.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 25,687,938 (97.99 MB)

 Trainable params: 2,100,226 (8.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [14]:
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import load_img

In [15]:
class_labels = list(validation_generator.class_indices.keys())
def classifica_imagem(imagem):
    img = load_img(imagem, target_size = (224, 224))
    img = image.img_to_array(img)
    img = np.expand_dims(img_, axis = 0)
    img = preprocessing_input(img_)
    preds = modelo.predict(img_)
    classe_predita = np.argmax(preds, axis = 1)[0]
    rotulo_predito = class_labels[classe_predita]
    print(f"Classe predita: {rotulo_predito} - {preds[0][classe_predita]:.4}")
    

In [16]:
history = modelo.fit(
    train_generator,
    steps_per_epoch = train_generator.samples // train_generator.batch_size,
    validation_data = validation_generator,
    validation_steps = validation_generator.samples // validation_generator.batch_size,
    epochs = 10
)

/home/turma02/anaconda3/envs/turma02/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


I0000 00:00:1736890936.666790   19720 service.cc:148] XLA service 0x78be78013100 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1736890936.666988   19720 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 6GB Laptop GPU, Compute Capability 8.6
2025-01-14 17:42:17.008735: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1736890938.691321   19720 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-01-14 17:42:18.930788: W external/local_xla/xla/service/gpu/nvptx_compiler.cc:930] The NVIDIA driver's CUDA version is 12.4 which is older than the PTX compiler version 12.5.82. Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
202

 2/76 ━━━━━━━━━━━━━━━━━━━━ 4s 63ms/step - accuracy: 0.7000 - loss: 0.6389  

I0000 00:00:1736890945.327061   19720 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


65/76 ━━━━━━━━━━━━━━━━━━━━ 7s 721ms/step - accuracy: 0.8771 - loss: 0.4454 

2025-01-14 17:43:14.030810: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_4989_0', 8 bytes spill stores, 8 bytes spill loads



76/76 ━━━━━━━━━━━━━━━━━━━━ 90s 1s/step - accuracy: 0.8885 - loss: 0.4016 - val_accuracy: 0.9941 - val_loss: 0.0119
Epoch 2/10
 1/76 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - accuracy: 1.0000 - loss: 1.1206e-06

/home/turma02/anaconda3/envs/turma02/lib/python3.10/site-packages/keras/src/trainers/epoch_iterator.py:107: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


76/76 ━━━━━━━━━━━━━━━━━━━━ 11s 150ms/step - accuracy: 1.0000 - loss: 1.1206e-06 - val_accuracy: 1.0000 - val_loss: 0.0064
Epoch 3/10
76/76 ━━━━━━━━━━━━━━━━━━━━ 66s 865ms/step - accuracy: 0.9940 - loss: 0.0209 - val_accuracy: 0.9882 - val_loss: 0.0226
Epoch 4/10
76/76 ━━━━━━━━━━━━━━━━━━━━ 11s 142ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 0.9882 - val_loss: 0.0544
Epoch 5/10
76/76 ━━━━━━━━━━━━━━━━━━━━ 65s 860ms/step - accuracy: 0.9845 - loss: 0.0387 - val_accuracy: 0.9941 - val_loss: 0.0239
Epoch 6/10
76/76 ━━━━━━━━━━━━━━━━━━━━ 11s 145ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 0.9941 - val_loss: 0.0220
Epoch 7/10
76/76 ━━━━━━━━━━━━━━━━━━━━ 66s 875ms/step - accuracy: 0.9979 - loss: 0.0118 - val_accuracy: 1.0000 - val_loss: 0.0025
Epoch 8/10
76/76 ━━━━━━━━━━━━━━━━━━━━ 11s 152ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0030
Epoch 9/10
76/76 ━━━━━━━━━━━━━━━━━━━━ 55s 725ms/step - accuracy: 1.0000 - loss: 0.0011 - val

In [17]:
modelo.save('resnet_10_epoch_potato_chips.keras')

In [18]:
imagem = path + 'amostra1.jpeg'

In [19]:
path

'/home/turma02/lgacademy/potatochips/potato_chips/'

In [20]:
classifica_imagem(imagem)

NameError: name 'img_' is not defined